In [1]:
from typing_extensions import TypedDict, Literal, Annotated, Dict 
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.types import Command
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage
import os
import json
import subprocess

In [3]:
SYSTEM = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, don't explain. If a task has been completed, output STOP."
""
MAX_ITER = 5

WORKDIR = os.getcwd()

def safe_path(path):
    full_path = os.path.abspath(os.path.join(WORKDIR, path))
    if not full_path.startswith(WORKDIR):
        raise ValueError("Unsafe path detected")
    return full_path

@tool
def run_bash(command : str) -> str:
    """
    Run a bash command and return the output.
    """
    dangerous_commands = ["rm", "sudo", "shutdown", "reboot"]
    if any(dangerous_command in command for dangerous_command in dangerous_commands):
        return "Error: Command contains potentially dangerous operations. Aborting execution."
    try:
        result = subprocess.run(command, shell=True, cwd=os.getcwd(), text=True, capture_output=True, timeout=120)
        out = (result.stdout + result.stderr).strip()
        return out[:10000] if out else "(No output)"
    except subprocess.CalledProcessError as e:
        return f"Error: {e.stderr}"

@tool 
def run_read(path, limit=None):
    """
    Read the contents of a file.
    """
    lines = safe_path(path).read_text().splitlines()
    if limit:
        lines = lines[:limit]
    return "\n".join(lines)

@tool
def run_write(path, content):
    """
    Write content to a file.
    """
    safe_path(path).write_text(content)
    return f"Wrote {len(content)} bytes to {path}"

@tool
def run_edit(path, old_text, new_text):
    """
    Edit a file by replacing the first occurrence of old_text with new_text.
    """
    text = safe_path(path).read_text()
    if old_text not in text:
        return "Error: text not found"
    safe_path(path).write_text(text.replace(old_text, new_text, 1))
    return f"Edited {path}"

@tool
def run_glob(pattern):
    """
    Glob for files matching a pattern.
    """
    import glob as g
    return "\n".join(g.glob(pattern, root_dir=WORKDIR))

def messages_reducer(left : list, right : list) -> list:
    return left + right

class State(TypedDict):
    messages : Annotated[list[Dict[str, str]], messages_reducer]
    max_iterations : int

def call_function(state: State) -> Command:
    all_messages = state.get("messages", [])
    #print("All Messages:", all_messages)
    if state.get("max_iterations", 0) <= 0 or not all_messages:
        if not all_messages:
            mess = f"No messages to process. Stopping execution."
        else:
            mess = f"Reached maximum iterations. Stopping execution."
        return Command(
            update={"messages": [AIMessage(content=mess)]},
            goto=END
        )
    
    last_message = all_messages[-1]

    content = last_message.content if hasattr(last_message, "content") else last_message.get("content", "")

    if "stop" in content.strip().lower():
        return Command(
            update={"messages": [AIMessage(content="Received STOP signal. Ending execution.")]},
            goto=END
        )
    elif "stop" not in content.strip().lower() and state.get("max_iterations", 0) > 0:        
        model = ChatOllama(model="gemma4", temperature=0.1)
        agent = create_agent(model=model, system_prompt=SYSTEM, tools=[run_bash])
        result = agent.invoke(state)
        #print(result)
        return Command(
            update={
                "messages": result.get("messages", []),
                "max_iterations": state.get("max_iterations", 0) - 1
            },
            goto="call_function"
        )
    else:
        return Command(
            goto=END
        )
  
builder = StateGraph(State)
builder.add_node("call_function", call_function)
builder.add_edge(START, "call_function")

# builder.add_edge(START, "call_function")
# builder.add_edge("call_function", END)

graph = builder.compile()

initial_state = {
    "messages": [
        {"role": "user", "content": 'Create a file called hello.py that prints "Hello, World!"'}
    ],
    "max_iterations": MAX_ITER
}

result2 = graph.invoke(initial_state)
print("Final State:", result2)

Final State: {'messages': [{'role': 'user', 'content': 'Create a file called hello.py that prints "Hello, World!"'}, HumanMessage(content='Create a file called hello.py that prints "Hello, World!"', additional_kwargs={}, response_metadata={}, id='75d7a1f4-4d0e-491c-a5a5-e6a352c3a8c9'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-05-23T14:20:27.8543614Z', 'done': True, 'done_reason': 'stop', 'total_duration': 140593878100, 'load_duration': 63003406300, 'prompt_eval_count': 132, 'prompt_eval_duration': 58822940800, 'eval_count': 74, 'eval_duration': 17723279600, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--019e5533-520d-7fa2-b3a5-0499ba03e4ba-0', tool_calls=[{'name': 'run_bash', 'args': {'command': 'echo \'print("Hello, World!")\' > hello.py'}, 'id': '6fbae822-2aa8-4bf1-b8df-328b1fa3a87e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 74, '

In [8]:
import os
import json
import subprocess
from typing import Dict, List

# Optimized system prompt to enforce cleaner JSON and explicit completion handling
SYSTEM = f"""You are a coding agent at {os.getcwd()}. Use bash to solve tasks.
You must interact with tools using valid JSON format. 

If you need to run a command, respond ONLY with a JSON array containing a tool_call block.
Example:
[
    {{"type": "tool_call", "tool": "bash", "input": {{"command": "echo 'hello'"}}}}
]

If the task has been completed and no more actions are needed, output exactly: STOP
Act, don't explain your reasoning. Do not wrap your JSON response in markdown code blocks."""

TOOLS = [{
    "name": "bash",
    "description": "Run a shell command.",
    "input_schema": {
        "type": "object",
        "properties": {"command": {"type": "string"}},
        "required": ["command"],
    },
}]

def run_bash(command: str) -> str:
    """Run a bash command and return the output."""
    dangerous_commands = ["rm", "sudo", "shutdown", "reboot"]
    if any(dangerous_command in command for dangerous_command in dangerous_commands):
        return "Error: Command contains potentially dangerous operations. Aborting execution."
    try:
        result = subprocess.run(command, shell=True, cwd=os.getcwd(), text=True, capture_output=True, timeout=120)
        out = (result.stdout + result.stderr).strip()
        return out if out else "(Command executed successfully with no output)"
    except Exception as e:
        return f"Error executing command: {str(e)}"

def agent_call(messages: List[Dict[str, str]]):
    max_iter = 3  # Increased slightly to allow a final stop iteration
    
    # Initialize Ollama model once outside the loop
    model = ChatOllama(model="gemma4", temperature=0.1)
    
    while max_iter > 0:
        # Build a structured history string for LLMs that accept raw string inputs
        # Better yet, pass messages as objects if your framework supports it.
        history_str = ""
        for m in messages:
            history_str += f"\n{m['role'].capitalize()}: {m['content']}"

        prompt = f"System: {SYSTEM}\nTools: {str(TOOLS)}\nHistory: {history_str}\n\nAssistant:"
        
        response = model.invoke(prompt)
        content = response.content if hasattr(response, "content") else response.get("content", "")
        content = content.strip()
        
        print(f"\n--- Iteration {4 - max_iter} ---")
        print("Agent Response Content:\n", content)
        
        if "stop" in content.lower():
            print("Stop signal received. Exiting loop.")
            messages.append({"role": "assistant", "content": "STOP"})
            break
        
        # Robust string cleaning for JSON parsing
        content_cleaned = content.replace("\xa0", " ")
        if content_cleaned.startswith("```"):
            # Handles ```json ... ``` formatting fallback safely
            content_cleaned = content_cleaned.split("\n", 1)[1] if "\n" in content_cleaned else content_cleaned
            content_cleaned = content_cleaned.rsplit("```", 1)[0].strip()
        else:
            content_cleaned = content_cleaned.strip()
            
        try: 
            blocks = json.loads(content_cleaned)
        except json.JSONDecodeError as e:
            print(f"JSON Parsing failed. Raw: {content_cleaned}")
            return f"Error: LLM did not return valid JSON. Error: {str(e)}. Output was: {content}"
        
        # Append the assistant's intent to the message history
        messages.append({"role": "assistant", "content": content_cleaned})
        
        # Execute tool calls and append results immediately
        tool_outputs = []
        for block in blocks:
            if block.get("type") == "tool_call" and block.get("tool") == "bash":
                command = block.get("input", {}).get("command", "")
                print(f"Executing Bash: {command}")
                output = run_bash(command)
                tool_outputs.append(f"Tool (bash) output: {output}")
        
        # Combine current loop's outputs into a user-role status update for the next loop
        if tool_outputs:
            combined_output = "\n".join(tool_outputs)
            messages.append({"role": "user", "content": combined_output})
        else:
            messages.append({"role": "user", "content": "No tools were executed. Please continue or STOP."})
            
        max_iter -= 1
        
    return messages

# Run the fixed agent
history = [{"role": "user", "content": 'Create a file called hello.py that prints "Hello, World!"'}]
result = agent_call(history)


--- Iteration 1 ---
Agent Response Content:
 [
    {"type": "tool_call", "tool": "bash", "input": {"command": "echo 'print(\"Hello, World!\")' > hello.py"}}
]
Executing Bash: echo 'print("Hello, World!")' > hello.py

--- Iteration 2 ---
Agent Response Content:
 STOP
Stop signal received. Exiting loop.
